# Compare runs with paired-bootstrap intervals

Bootstrap saved predictions; do not refit models. Every metric within a cohort shares the same object draws. Keep native/controlled reliability results and the point-estimate holdout explicitly identified. Optional model-to-model comparisons pair by object ID.


In [ ]:
CONFIG_PATH = "configs/synthetic-optuna.yaml"
RUN_DIR = "runs/synthetic-optuna"
SHOW_PLOTS = True
RUN_DIRS = []  # optional additional prepared/evaluated runs
REFERENCE_RUN_DIR = ""  # optional paired zML comparison, e.g. the with-i run


In [ ]:
from pathlib import Path
from IPython.display import display
from turbopdz.workflow import run_stage, load_run

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "pyproject.toml").exists()), Path.cwd())
config_path = (PROJECT_ROOT / CONFIG_PATH).resolve()
run_dir = (PROJECT_ROOT / RUN_DIR).resolve()
print(f"config={config_path}\nrun={run_dir}")


In [ ]:
import pandas as pd
from turbopdz.evaluation.reporting import write_tables, compare_point_runs

paths = [run_dir] + [(PROJECT_ROOT / path).resolve() for path in RUN_DIRS]
results = []
for path in paths:
    _, _, config = load_run(path)
    from turbopdz.workflow import bootstrap_run
    summary = run_stage("bootstrap", config_path, path) if path == run_dir else bootstrap_run(path)
    write_tables(summary, path / "tables")
    results.append(summary.assign(run=path.name))
comparison = pd.concat(results, ignore_index=True)
display(comparison)
comparison.to_csv(run_dir / "comparison.csv", index=False)
from turbopdz.analysis import write_bootstrap_diagnostics
display(write_bootstrap_diagnostics(run_dir, show=SHOW_PLOTS))


In [ ]:
if REFERENCE_RUN_DIR:
    from turbopdz.config import load_config
    config = load_config(config_path)
    reference_dir = (PROJECT_ROOT / REFERENCE_RUN_DIR).resolve()
    paired = compare_point_runs(pd.read_csv(run_dir / "point_holdout.csv", dtype={"object_id": str}),
                                pd.read_csv(reference_dir / "point_holdout.csv", dtype={"object_id": str}),
                                n_boot=config["bootstrap"]["resamples"],
                                confidence_level=config["bootstrap"]["confidence_level"], seed=config["seed"])
    write_tables(paired.summary, run_dir / "paired-model-comparison")
    display(paired.summary)
